In [130]:
from pydantic import BaseModel, Field, ConfigDict

In [ ]:
class Address(BaseModel):
    street: str = Field(max_length=100)
    state: str = Field(default="MA", max_length=2, description="State must be a 2-letter abbreviation")
    zip_code: str

class User(BaseModel):
    name: str = Field(min_length=4, max_length=50, description="Name must be between 4 and 50 characters")
    age: int = Field(gt=18, lt=120, description="Age must be between 18 and 120")
    email: str = Field(pattern=r'^\S+@\S+\.\S+$', description="Email must be a valid email address")
    address: list[Address] = Field(default_factory=Address, description="User's address")
    contact_numbers: list[str] = Field(default_factory=list, description="List of contact numbers", min_items=1)


/var/folders/75/9jzh8kz52fb0g16_q61mthm00000gn/T/ipykernel_34744/405021146.py:11: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  contact_numbers: list[str] = Field(default_factory=list, description="List of contact numbers", min_items=1)


In [ ]:
json_str = """
{
    "name": "John Doe",
    "age": 30,
    "email": "john.doe@example.com",
    "address": [{
        "street": "123 Main St",
        "state": "MA",
        "zip_code": "12345"
    }],
    "contact_numbers": ["123-456-7890"]
}
""" 

In [159]:
user = User.model_validate_json(json_str)

In [161]:
print(user.contact_numbers.append("555-555-5555"))
print(user.address)

None
[Address(street='123 Main St', state='MA', zip_code='12345')]


In [152]:
user.model_dump()

{'name': 'John Doe',
 'age': 30,
 'email': 'john.doe@example.com',
 'address': {'street': '123 Main St', 'state': 'MA', 'zip_code': '12345'},
 'contact_numbers': ['123-456-7890', '555-555-5555']}

In [154]:
User.model_json_schema()

{'$defs': {'Address': {'properties': {'street': {'maxLength': 100,
     'title': 'Street',
     'type': 'string'},
    'state': {'default': 'MA',
     'description': 'State must be a 2-letter abbreviation',
     'maxLength': 2,
     'title': 'State',
     'type': 'string'},
    'zip_code': {'title': 'Zip Code', 'type': 'string'}},
   'required': ['street', 'zip_code'],
   'title': 'Address',
   'type': 'object'}},
 'properties': {'name': {'description': 'Name must be between 4 and 50 characters',
   'maxLength': 50,
   'minLength': 4,
   'title': 'Name',
   'type': 'string'},
  'age': {'description': 'Age must be between 18 and 120',
   'exclusiveMaximum': 120,
   'exclusiveMinimum': 18,
   'title': 'Age',
   'type': 'integer'},
  'email': {'description': 'Email must be a valid email address',
   'pattern': '^\\S+@\\S+\\.\\S+$',
   'title': 'Email',
   'type': 'string'},
  'address': {'$ref': '#/$defs/Address', 'description': "User's address"},
  'contact_numbers': {'description': 'Lis

## Custom Validations, Computed Fields

In [ ]:
from pydantic import BaseModel, Field, field_validator, SecretStr, model_validator

In [6]:
class User(BaseModel):
    username: str
    password: SecretStr

    @field_validator('username')
    def validate_username(cls, name):
        if ' ' in name:
            raise ValueError("Name cannot contain spaces")
        return name.lower()

In [9]:
username = User(username="shivam1892", password="secret123")
print(username.username)  # Output: shivam1892
print(username.password)  # Output: SecretStr('secret123')
print(username.password.get_secret_value())  # Output: secret123

shivam1892
**********
secret123


In [31]:
from pydantic import BaseModel, field_validator, model_validator, computed_field

class JobApplication(BaseModel):
    full_name: str
    email: str
    years_experience: int
    password: str
    confirm_password: str
    domain: str | None = None

    @field_validator("full_name", mode="after")
    @classmethod
    def normalize_name(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("full_name cannot be empty")
        return cleaned.title()

    @field_validator("years_experience", mode="before")
    @classmethod
    def strip_years_suffix(cls, value):
        # handles messy input like "5 years" arriving as raw text
        if isinstance(value, str):
            digits = "".join(ch for ch in value if ch.isdigit())
            return int(digits) if digits else value
        return value

    @model_validator(mode="after")
    def passwords_must_match(self):
        if self.password != self.confirm_password:
            raise ValueError("password and confirm_password do not match")
        return self

    @computed_field
    @property
    def experience_tier(self) -> str:
        if self.years_experience < 2:
            return "junior"
        elif self.years_experience < 7:
            return "mid"
        return "senior"

app = JobApplication(
    full_name="  Shivam Pasricha  ", 
    email="shivam@gmal.com", 
    years_experience="5 years",
    password="securepassword",
    confirm_password="securepassword"
)
print(app)   # Shivam Pasricha 5

full_name='Shivam Pasricha' email='shivam@gmal.com' years_experience=5 password='securepassword' confirm_password='securepassword' domain=None experience_tier='mid'


## Serialization control: exclude, include, exclude_unset

In [27]:
app.model_dump(exclude={"password", 'confirm_password'})

{'full_name': 'Shivam Pasricha',
 'email': 'shivam@gmal.com',
 'years_experience': 5,
 'experience_tier': 'mid'}

In [28]:
app.model_dump(include={"full_name", 'email'})

{'full_name': 'Shivam Pasricha', 'email': 'shivam@gmal.com'}

In [ ]:
app.model_dump(exclude_unset=True) # domain will not be dumped since it was not set

{'full_name': 'Shivam Pasricha',
 'email': 'shivam@gmal.com',
 'years_experience': 5,
 'password': 'securepassword',
 'confirm_password': 'securepassword',
 'experience_tier': 'mid'}

In [ ]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    state: str
    pin_code: str

class Applicant(BaseModel):
    name: str
    email: str
    address: Address        # a whole model, used as a field type

# Parsing straight from a nested dictionary — the common real-world case
incoming = {
    "name": "Rohan Mehta",
    "email": "rohan@example.com",
    "address": {"city": "Pune", "state": "Maharashtra", "pin_code": "411001"},
}
applicant = Applicant.model_validate(incoming)
print(applicant.address.city)   # "Pune" — dot-chain access, fully typed

# Lists of nested models work the same way
class WorkExperience(BaseModel):
    company: str
    role: str
    years: int

class Application(BaseModel):
    applicant: Applicant
    work_history: list[WorkExperience]

## Pydantic Settings
```uv add pydantic-settings```

Useful to knowing about all environment varable available on booting up the application

<b> The problem with os.getenv() </b>
- Every value from os.getenv() is a string — or None — always. 
- MAX_CONNECTIONS arrives as the string "200", not the number 200. 
- DEBUG arrives as the string "true", not the boolean True.

we need to write logic to check if the value exists, convert its type, validate it, handle the default. More the number of configuration values and messy & repetitive risk

### BaseSettings, .env files, and SecretStr
- BaseSettings is used same as Basemodel, Field names map to environment variable names automatically, uppercased: api_key looks for API_KEY.
- SettingsConfigDict(env_file=".env") tells Pydantic to also read from a .env file. Every constraint e.g. (Field(ge=1, le=1000)) applies here. SecretStr keeps API keys and passwords out of logs and stack traces by default.

In [ ]:
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    api_key: SecretStr
    max_connections: int = Field(default=20, ge=1, le=200)
    debug: bool = False

settings = AppSettings()   # reads from .env / environment automatically
print(settings.max_connections, type(settings.max_connections))   # 20 <class 'int'>
print(settings.api_key)                          # ********** (masked)
print(settings.api_key.get_secret_value())       # the real value, on purpose

## Pydantic in with FastAPI

### Automatic request validation
- FastAPI endpoint can take a Pydantic model directly as a parameter type. 
- It parses the incoming JSON request body, validates every field using the model, and rejects malformed requests with a detailed error before the endpoint function's body start to execute. 
- No manual validations and type casting required
- In below example Applicant model is mapped with request body of endpoint

### Response models and auto-generated docs
* ApplicationReceipt is mapped with Response of endpoint `response_model=ApplicationReceipt`

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr, Field
from pydantic import computed_field

app = FastAPI()

class Applicant(BaseModel):
    full_name: str = Field(min_length=2, max_length=100)
    email: EmailStr
    years_experience: int = Field(ge=0, le=50)

class ApplicationReceipt(BaseModel):
    full_name: str
    years_experience: int

    @computed_field
    @property
    def tier(self) -> str:
        return "senior" if self.years_experience >= 7 else "standard"

@app.post("/apply", response_model=ApplicationReceipt)
def submit(applicant: Applicant) -> ApplicationReceipt:
    return ApplicationReceipt(
        full_name=applicant.full_name,
        years_experience=applicant.years_experience,
    )


## LLMs Response Formating and Validations
<b>Issue with raw LLM text output</b>
An LLM asked to "extract product info as JSON" 
- It wrap the response in a markdown code fence
- Add a friendly preamble ("Sure! Here's the extracted info:")
- Return a number formatted as a string one run and an int the next. 

All are realistic, common outputs and only one successfully parsed with json.loads() call unmodified.
- Building string-cleanup logic can break the moment response format changes. 
- The real fix is asking the provider to *guarantee* the shape, instead of hoping and cleaning up after the fact.

In [ ]:
import json
from pydantic import BaseModel, ValidationError

class ProductInfo(BaseModel):
    name: str
    price: float
    in_stock: bool

# Four realistic, unmodified LLM responses to the SAME prompt:
responses = [
    '{"name": "MacBook Pro", "price": 1999.0, "in_stock": true}',                 # clean
    '```json\n{"name": "MacBook Pro", "price": 1999.0, "in_stock": true}\n```',  # markdown-wrapped
    'Sure! Here it is: {"name": "MacBook Pro", "price": 1999.0, "in_stock": true}', # chatty preamble
    '{"name": "MacBook Pro", "price": "1999 dollars", "in_stock": "yes"}',          # right shape, wrong types
]
print(responses)

['{"name": "MacBook Pro", "price": 1999.0, "in_stock": true}', '```json\n{"name": "MacBook Pro", "price": 1999.0, "in_stock": true}\n```', 'Sure! Here it is: {"name": "MacBook Pro", "price": 1999.0, "in_stock": true}', '{"name": "MacBook Pro", "price": "1999 dollars", "in_stock": "yes"}']


### OpenAI structured outputs
- OpenAI SDK accepts a Pydantic model. `client.responses.parse(text_format=Model)`.
- Returned response is already a validated instance of your class, not a string, not a raw dict.

In [ ]:
from openai import OpenAI
from pydantic import BaseModel

class ProductInfo(BaseModel):
    name: str
    price: float
    category: str
    in_stock: bool

client = OpenAI()
response = client.responses.parse(
    model="gpt-4o",
    input="Extract: The new MacBook Pro costs $1999, available now in Electronics.",
    text_format=ProductInfo,   # pass the MODEL, not a JSON schema dict
)

product: ProductInfo = response.output_parsed   # already validated
print(product.name, product.price, product.in_stock)

### Claude structured outputs
- Anthropic's `client.messages.parse(output_format=YourModel)` returns response.parsed_output
- ConfigDict(extra="forbid") tightens the generated JSON schema so model can't add extra keys that weren't asked



In [ ]:
from anthropic import Anthropic
from pydantic import BaseModel, ConfigDict

class ProductInfo(BaseModel):
    model_config = ConfigDict(extra="forbid")   # no surprise extra keys
    name: str
    price: float
    category: str
    in_stock: bool

client = Anthropic()
response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Extract: MacBook Pro, $1999, Electronics, in stock."}],
    output_format=ProductInfo,
)

product: ProductInfo = response.parsed_output   # already validated

### Literal for constrained outputs, and the retry pattern
- Literal physically restricts what values an AI classification is allowed to return..
- When a response fails validation, the resulting ValidationError is the trigger point for a self-correcting retry loop: feed the error message back to the model and ask it to try again. 

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

class TicketClassification(BaseModel):
    category: Literal["billing", "shipping", "technical", "general_question"]
    priority: Literal["low", "medium", "high"]
    sentiment: Literal["positive", "negative", "neutral"]
    summary: str = Field(max_length=200)

def classify_with_retry(raw_text: str, max_retries: int = 3):
    last_error = None
    for attempt in range(max_retries):
        raw_response = ask_llm(raw_text, previous_error=last_error)   # your LLM call
        try:
            return TicketClassification.model_validate_json(raw_response)
        except ValidationError as e:
            last_error = str(e)   # fed back into the NEXT prompt attempt
    raise RuntimeError("Failed after max retries")

In [ ]:
import re
from typing import Literal
from pydantic import BaseModel, Field, field_validator, model_validator, computed_field, ConfigDict

class IncomingTicket(BaseModel):
    message: str = Field(min_length=5, max_length=2000)
    customer_name: str | None = None

    @field_validator("message", mode="before")
    @classmethod
    def redact_emails(cls, value):
        if isinstance(value, str):
            return re.sub(r"[\w.-]+@[\w.-]+\.\w+", "[redacted-email]", value)
        return value

class CustomerInfo(BaseModel):
    name: str | None = None
    order_id: int | None = None

class TriagedTicket(BaseModel):
    model_config = ConfigDict(extra="forbid")
    category: Literal["billing", "shipping", "technical", "general_question"]
    priority: Literal["low", "medium", "high"]
    sentiment: Literal["positive", "negative", "neutral"]
    summary: str = Field(max_length=200)
    customer: CustomerInfo = Field(default_factory=CustomerInfo)

    @computed_field
    @property
    def sla_hours(self) -> int:
        return {"high": 4, "medium": 24, "low": 72}[self.priority]

    @model_validator(mode="after")
    def high_priority_needs_real_category(self):
        if self.priority == "high" and self.category == "general_question":
            raise ValueError("'high' priority cannot pair with 'general_question'")
        return self

In [ ]:
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")

    anthropic_api_key: SecretStr | None = None
    openai_api_key: SecretStr | None = None
    openrouter_api_key: SecretStr | None = None

    def active_provider(self) -> str:
        if self.anthropic_api_key: return "anthropic"
        if self.openai_api_key: return "openai"
        if self.openrouter_api_key: return "openrouter"
        raise RuntimeError("No AI provider key configured.")

In [ ]:
@app.post("/triage", response_model=TriageResponse)
def triage(incoming: IncomingTicket) -> TriageResponse:
    # incoming.message is already validated AND redacted by this point
    result = triage_ticket(incoming)   # calls Anthropic/OpenAI/OpenRouter,
                                        # returns an already-validated TriagedTicket
    return TriageResponse.from_triaged_ticket(result)